# Player Tracking in Sports - Multi-View Tracking and 3D Reconstruction

## Imports

In [ ]:
from src.utils.video import open_video, get_frames, produce_detection_output_video
from src.utils.visualization import show_images
from src.detection.mog2_detection import run_mog2_detection, mog2_to_detection_output

import cv2
import time

In [ ]:
CURRENT_CAMERA_ID = "cam_13"

## Path Definitions

In [ ]:
VIDEOS_DIR = "data/videos"
CAMERA_SETTINGS_DIR = "data/camera_settings"

# Define fundamental paths for each camera
CAMERAS = {
    "cam_13": {
        "video_path": f"{VIDEOS_DIR}/out13.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam13_settings.json",
    },
    "cam_2": {
        "video_path": f"{VIDEOS_DIR}/out2.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam2_settings.json",
    },
    "cam_4": {
        "video_path": f"{VIDEOS_DIR}/out4.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam4_settings.json",
    },
}

## Open Video and Read Frames

In [ ]:
# Currently just one camera
cap = open_video(CAMERAS[CURRENT_CAMERA_ID]["video_path"])
frames_color, _ = get_frames(cap, max_frames=150)  # Load only the first 150 frames for testing
fps = cap.get(cv2.CAP_PROP_FPS)

# Release the video capture object to free resources
cap.release()

show_images(frames_color)

## MOG2 Detection
We use the MOG2 (Mixture of Gaussians v2) background subtractor to detect moving players in each frame. The pipeline runs in a single per-frame loop: illumination normalization → MOG2 background subtraction → field-color suppression → morphological opening/closing → blob area filtering.

### Run MOG2 Detection on All Frames

**Output per frame:**
- Binary mask: foreground pixels (white = moving object, black = background)

All pipeline stages are fused in a single loop. Tune parameters as needed:
- `history_length`: how many past frames build the background model (higher = more stable)
- `var_threshold`: sensitivity to foreground/background separation (lower = more sensitive)
- `min_area` / `max_area`: blob area bounds to keep only player-sized regions

In [ ]:
print(f"Processing {len(frames_color)} frames with MOG2 detection...")
start_time = time.time()

masks = run_mog2_detection(frames_color)

end_time = time.time()
print(f"MOG2 detection complete")
print(f"Processing time: {end_time - start_time:.2f} seconds")

show_images(masks)

### Convert Masks to Detection Output

Convert the binary masks into a `DetectionOutput` — the same structured format produced by YOLO detection. Each connected component (blob) in a mask becomes one `Detection` with a bounding box.

In [ ]:
detection_output = mog2_to_detection_output(
    masks,
    camera_id=CURRENT_CAMERA_ID,
    fps=fps,
)

print(f"Total detections: {sum(f.num_detections for f in detection_output.frames)} across {len(detection_output.frames)} frames")

## Inspect Detections

In [ ]:
print(f"First frame detections: {detection_output.frames[0].num_detections} objects detected")
print(f"First frame detection details:")
for i, detection in enumerate(detection_output.frames[0].detections[:5]):
    print(f"  Detection {i+1}: Class={detection.class_name}, Confidence={detection.confidence:.2f}, BBox={detection.bbox}")

In [ ]:
produce_detection_output_video(frames_color, detection_output, f"results/detection/{CURRENT_CAMERA_ID}/mog2_detections.mp4")